# Foraging models: MVT (port level) and inference (patch level)

Two decision levels, two data levels:

| | Bpod only | + NaviGraph graph locations |
|---|---|---|
| **Port level** (repeat vs leave the port) | MVT threshold (fixed / free weights) vs count / depletion / win-stay / GLM | MVT-partner with measured travel time to the partner port |
| **Patch level** (stay in pair vs leave) | Bayesian observer, consecutive failures, leaky integrator, pair-value RL, MVT, combinations, GLM-HMM states | + aborted exits, prospective (expected) travel time |
| **Destination** after a patch leave | static graph distance, block order, recency, last outcome, RL value | measured transit time |

Every model is fitted **separately per subject** and compared by **leave-one-session-out** held-out log-likelihood
(bits per decision relative to a constant-rate model), held-out McFadden pseudo-R² and AUC. Hyper-parameters
(time constants, learning rates, hazard) are chosen on the training sessions only. All models of one comparison are
scored on exactly the same decisions.

Sections: 0 config · 1 graph · 2 sessions · 3 trial tables · 4 graph-derived variables · 5 port level · 6 patch level ·
6b model comparison · 7 destination · 8 graph-upgraded models · 9 MVT threshold test · 10 across sessions ·
11 GLM-HMM states · 12 save.

Figures are written to `Graphs/all_subjects/foraging_models/figures/` (PNG + PDF); tables to `Graphs/<subject>/foraging_models/`.

## 0. Configuration
Edit this cell only. `SUBJECTS` lists the mice to analyse; each needs an entry in `PORT_LAYOUTS`.

In [ ]:
from pathlib import Path

SUBJECTS = ['NPC1', 'NPC4']               # add '3L', '3NC' when ready (each needs an entry in PORT_LAYOUTS)
FPS = 40                                  # constant for all sessions
FRAME_S = 1.0 / FPS

NAVIGRAPH_RAW_ROOT = Path('/var/home/almogmeir/Documents/M.Sc/Project/Input/Navigraph_Raw_Data')
PIPELINE_ROOT      = Path('/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs')
GRAPH_PKL          = Path('/var/home/almogmeir/Documents/GitHub/NaviGraph/FullMazeGrid.pkl')
BPOD_DATA_ROOT     = Path('/var/home/almogmeir/Documents/M.Sc/Project/BPODData')
BPOD_PROTOCOL      = None                 # e.g. 'Adva_maze_pairs_fix' to pin the protocol folder
OUTPUT_ROOT        = Path('Graphs')       # tables -> Graphs/<subject>/foraging_models/, figures -> Graphs/all_subjects/foraging_models/figures/

# File names inside PIPELINE_ROOT/<subject>/<DD_MM_YYYY>/ (swap for the improved_/final_ versions if preferred)
FILES = {
    'reward':    'reward_df_annotated.csv',
    'traversal': 'traversal_df.csv',
    'node_edge': 'node_edge_df.csv',
}

# Analysis options
EXCLUDE_MAZE_A_PRIME = False   # True drops change-of-maze sessions from all models
P_ACTIVE, P_INACTIVE = 0.9, 0.1
RHO_TAUS_S = [10, 30, 60, 120, 300, 600, 1200, 2400, 4800]  # candidate time constants (s) for the running reward rate
MVT_ENV_TAUS_S = [t for t in RHO_TAUS_S if t >= 120]  # MVT 'environment' rate: windows longer than a patch visit
LEAK_ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]     # candidate learning rates for leaky integrators / choice history
HAZARD_SCALES = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]  # block-switch hazard as a multiple of 1 / mean block length
PORT_BELIEF_SCALE = 1.0                                 # hazard scale of the belief used by the port-level MVT
RL_ALPHAS = [0.1, 0.2, 0.3, 0.5, 0.7]                   # pair-value Q-learning: learning rate
RL_FORGET = [0.0, 0.01, 0.03, 0.1, 0.3, 0.5, 1.0]       # ... and per-contact forgetting of the other pairs towards 0.5 (1 = no memory)
HANDLING_S = 4.0                                        # added to residence time in the local patch reward rate
CONSUMPTION_S = 2.0                                     # Bpod consumption period, added to the graph travel time
GLMHMM_STATES = [1, 2, 3]                               # GLM-HMM state counts compared (1 = plain logistic regression)
GLMHMM_FOLDS = 5                                        # session-wise cross-validation folds for the GLM-HMM
GLMHMM_MAX_ITER = 60
PATH_LENGTH_FROM_PIPELINE = True          # use reward_df's transition lengths (your validated reconstruction) for path_len/excess
ZIGZAG_MAX_FRAMES = 1                     # collapse x -> y -> x excursions where y lasts <= this many frames
RIDGE = 1e-3                              # tiny L2 penalty for numerical stability in logistic fits
RANDOM_SEED = 0
SAVE_FIGURES = True
FIG_FORMATS = ('png', 'pdf')

In [ ]:
# ============================================================================
# Port layout — copied from the main pipeline (edit there and here together).
# ============================================================================
PORT_LAYOUTS = {
    'NPC1': [
        (None, '01/09/2025', {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L56', ['L43']), 4: ('L510', ['L45']),
            5: ('L521', ['L410']), 6: ('L525', ['L412']), 7: ('R55', ['R42']), 8: ('R59', ['R44'])}),
        ('02/09/2025', '06/09/2025', {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L56', ['L43']), 4: ('L510', ['L45']),
            5: ('L521', ['L410']), 6: ('L518', ['L49']), 7: ('R55', ['R42']), 8: ('R59', ['R44'])}),
        ('07/09/2025', '08/09/2025', {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L56', ['L43']), 4: ('L510', ['L45']),
            5: ('L521', ['L410']), 6: ('L518', ['L49']), 7: ('R59', ['R44']), 8: ('R55', ['R42'])}),
        ('09/09/2025', None, {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L56', ['L43']), 4: ('L510', ['L45']),
            5: ('L521', ['L410']), 6: ('L525', ['L412']), 7: ('R59', ['R44']), 8: ('R55', ['R42'])}),
    ],
    'NPC4': [
        ('01/03/2026', '16/06/2026', {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L510', ['L45']), 4: ('L56', ['L43']),
            5: ('L521', ['L410']), 6: ('L525', ['L412']), 7: ('R59', ['R44']), 8: ('R55', ['R42'])}),
        ('17/06/2026', '18/06/2026', {
            1: ('R521', ['R410']), 2: ('R526', ['R413']), 3: ('L510', ['L45']), 4: ('L56', ['L43']),
            5: ('L530', ['L415']), 6: ('L517', ['L48']), 7: ('R59', ['R44']), 8: ('R55', ['R42'])}),
        ('19/06/2026', None, {
            1: ('R522', ['R411']), 2: ('R526', ['R413']), 3: ('L510', ['L45']), 4: ('L56', ['L43']),
            5: ('L521', ['L410']), 6: ('L525', ['L412']), 7: ('R59', ['R44']), 8: ('R55', ['R42'])}),
    ],
}
PORT_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8)]
PORT_PAIRS_BY_SUBJECT = {}

# Maze A' (change of maze) = layout periods whose leaf set differs from the subject's first period.
# Set explicitly here to override the automatic rule.
MAZE_A_PRIME_OVERRIDE = {}   # e.g. {'NPC1': [('02/09/2025', '08/09/2025')]}

## Imports and helpers

In [ ]:
import ast, glob, re, warnings
import numpy as np
import pandas as pd
import networkx as nx
import scipy.io as sio
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from scipy.stats import rankdata, spearmanr
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
pd.set_option('display.width', 200)
rng = np.random.default_rng(RANDOM_SEED)
try:
    display
except NameError:
    display = print

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.frameon': False, 'legend.fontsize': 7, 'figure.dpi': 110})
SUBJECT_COLORS = dict(zip(SUBJECTS, plt.cm.tab10.colors))


def ensure_dir(path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    return path


def _date_key(d):
    return None if d is None else (d[6:10], d[3:5], d[0:2])


def resolve_port_layout(subject, session):
    """session = 'DD/MM/YYYY' -> {port: (leaf, [parents])}."""
    s = _date_key(session)
    for lo_, hi_, layout in PORT_LAYOUTS[subject]:
        lo, hi = _date_key(lo_), _date_key(hi_)
        if (lo is None or lo <= s) and (hi is None or s <= hi):
            return layout
    raise KeyError(f'No port layout period covers {subject} on {session}')


def is_maze_a_prime(subject, session):
    s = _date_key(session)
    if subject in MAZE_A_PRIME_OVERRIDE:
        return any(_date_key(a) <= s <= _date_key(b) for a, b in MAZE_A_PRIME_OVERRIDE[subject])
    base = {leaf for leaf, _ in PORT_LAYOUTS[subject][0][2].values()}
    cur = {leaf for leaf, _ in resolve_port_layout(subject, session).values()}
    return cur != base


def pairs_for(subject):
    return PORT_PAIRS_BY_SUBJECT.get(subject, PORT_PAIRS)


def pair_of_port(subject):
    return {p: i for i, pr in enumerate(pairs_for(subject)) for p in pr}


N_PAIRS = max(len(pairs_for(s)) for s in SUBJECTS)


def fit_logit(X, y, w=None):
    """Logistic regression with intercept (column of ones added) and tiny ridge. Returns beta."""
    X1 = np.column_stack([np.ones(len(X)), X])
    w = np.ones(len(y)) if w is None else w
    def nll(b):
        z = X1 @ b
        return -(w * (y * z - np.logaddexp(0, z))).sum() + RIDGE * (b[1:] ** 2).sum()
    def grad(b):
        p = expit(X1 @ b)
        g = -X1.T @ (w * (y - p))
        g[1:] += 2 * RIDGE * b[1:]
        return g
    res = minimize(nll, np.zeros(X1.shape[1]), jac=grad, method='L-BFGS-B')
    return res.x


def logit_ll(beta, X, y):
    z = np.column_stack([np.ones(len(X)), X]) @ beta
    return (y * z - np.logaddexp(0, z))


def auc(y, p):
    """Area under the ROC curve (Mann-Whitney U), ignoring rows without a prediction."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok].astype(bool), p[ok]
    n1, n0 = y.sum(), (~y).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    return (rankdata(p)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def as_grid(make, df):
    """Evaluate a model spec: {param: X} for a hyper-parameter grid, or a single X (param None)."""
    g = make(df)
    g = g if isinstance(g, dict) else {None: g}
    return {p: np.asarray(X, float).reshape(len(df), -1) for p, X in g.items()}


def most_common_param(per, model):
    vc = pd.Series(per[f'par_{model}'].tolist(), dtype=object).dropna().value_counts()
    return vc.index[0] if len(vc) else None


def loso_compare(df, y_col, models, group_col='session', min_test=5):
    """Leave-one-session-out comparison of logistic models.

    models: {name: callable(df) -> X (n x k) or {param: X}}; grid parameters are chosen on the training sessions.
    All models are scored on the same rows (rows with a non-finite feature in any model are dropped, and reported).
    Returns (summary, per-session table, held-out predictions [one column per model + 'constant'])."""
    grids = {m: as_grid(make, df) for m, make in models.items()}
    ok = np.ones(len(df), bool)
    for g in grids.values():
        for X in g.values():
            ok &= np.isfinite(X).all(1)
    if (~ok).any():
        print(f'  loso_compare: {(~ok).sum()} of {len(df)} rows dropped (non-finite features)')
    y, grp = df[y_col].values.astype(float), df[group_col].values
    pred = pd.DataFrame(np.nan, index=df.index, columns=['constant'] + list(models))
    rows = []
    for s in pd.unique(grp[ok]):
        te, tr = ok & (grp == s), ok & (grp != s)
        if te.sum() < min_test or len(np.unique(y[tr])) < 2:
            continue
        p0 = np.clip(y[tr].mean(), 1e-6, 1 - 1e-6)
        pred.loc[te, 'constant'] = p0
        rec = {'session': s, 'n': int(te.sum()),
               'll_const': (y[te] * np.log(p0) + (1 - y[te]) * np.log(1 - p0)).sum()}
        for m, g in grids.items():
            best = None
            for par, X in g.items():
                b = fit_logit(X[tr], y[tr])
                ll_tr = logit_ll(b, X[tr], y[tr]).sum()
                if best is None or ll_tr > best[0]:
                    best = (ll_tr, par, b, X)
            _, par, b, X = best
            rec[f'll_{m}'], rec[f'par_{m}'] = logit_ll(b, X[te], y[te]).sum(), par
            pred.loc[te, m] = expit(np.column_stack([np.ones(te.sum()), X[te]]) @ b)
        rows.append(rec)
    per = pd.DataFrame(rows)
    n, ll0 = per['n'].sum(), per['ll_const'].sum()
    summary = pd.DataFrame({
        'model': list(models),
        'bits_per_decision': [(per[f'll_{m}'] - per['ll_const']).sum() / n / np.log(2) for m in models],
        'pseudo_R2': [(per[f'll_{m}'].sum() - ll0) / -ll0 for m in models],
        'AUC': [auc(y, pred[m].values) for m in models],
        'sessions_better_than_const': [int((per[f'll_{m}'] > per['ll_const']).sum()) for m in models],
        'n_sessions': len(per),
        'n_decisions': n,
        'param_mode': [most_common_param(per, m) for m in models],
    }).sort_values('bits_per_decision', ascending=False).reset_index(drop=True)
    return summary, per, pred


def session_bits(per, model, base='ll_const'):
    """Held-out bits per decision of one model in each session, relative to the baseline column."""
    return (per[f'll_{model}'] - per[base]) / per['n'] / np.log(2)


def session_dates(sessions):
    return pd.to_datetime(pd.Series(sessions).str.split().str[1], format='%d/%m/%Y').values


# ---------------------------------------------------------------------------- figures
def save_fig(fig, name, subject='all_subjects'):
    if not SAVE_FIGURES:
        return
    out = OUTPUT_ROOT / subject / 'foraging_models' / 'figures'
    out.mkdir(parents=True, exist_ok=True)
    for ext in FIG_FORMATS:
        fig.savefig(out / f'{name}.{ext}', dpi=200, bbox_inches='tight')


def plot_model_comparison(ax, per, models, color, title='', base='ll_const', n_boot=2000):
    """Held-out bits/decision vs baseline: grey line per session across models; dot = decision-weighted mean
    (= pooled value in the tables) with a 95% bootstrap CI over sessions."""
    B = np.column_stack([session_bits(per, m, base) for m in models])
    w = per['n'].values.astype(float)
    x = np.arange(len(models))
    for row in B:
        ax.plot(x, row, '-', c='0.85', lw=0.6, zorder=1)
    ax.scatter(np.tile(x, len(B)), B.ravel(), s=5, c='0.65', zorder=2)
    mean = (B * w[:, None]).sum(0) / w.sum()
    idx = rng.integers(0, len(B), (n_boot, len(B)))
    boot = np.einsum('bs,bsm->bm', w[idx], B[idx]) / w[idx].sum(1, keepdims=True)
    lo, hi = np.percentile(boot, [2.5, 97.5], 0)
    ax.errorbar(x, mean, yerr=[mean - lo, hi - mean], fmt='o', c=color, ms=5, capsize=3, zorder=3)
    ax.axhline(0, c='k', lw=0.6, ls='--')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=55, ha='right', fontsize=7)
    ax.set_ylabel('held-out bits / decision\nvs baseline')
    ax.set_title(title)


def plot_binned(ax, x, y, sessions, bins=None, preds=None, color='k', label='data', min_n=20):
    """Data: mean of y in bins of x (bins=None: each distinct x), error bars = SEM across sessions.
    preds: {label: (held-out probability, colour)} drawn as lines (mean prediction in the same bins)."""
    df = pd.DataFrame({'x': np.asarray(x, float), 'y': np.asarray(y, float), 's': np.asarray(sessions)})
    df['b'] = df['x'] if bins is None else pd.qcut(df['x'], bins, duplicates='drop')
    g = df.groupby('b', observed=True)
    size = g.size()
    keep = size[size >= min_n].index
    xm = g['x'].mean()[keep].values
    sem = df.groupby(['b', 's'], observed=True)['y'].mean().groupby(level=0, observed=True).sem().reindex(keep).values
    ax.errorbar(xm, g['y'].mean()[keep].values, yerr=sem, fmt='o', ms=4, c=color, capsize=2, label=label, zorder=3)
    for lab, (p, c) in (preds or {}).items():
        df['p'] = np.asarray(p, float)
        ax.plot(xm, df.groupby('b', observed=True)['p'].mean()[keep].values, '-', c=c, lw=1.5, label=lab, zorder=2)


def shade_maze_prime(ax, x, is_prime):
    """Grey background behind change-of-maze (A') sessions."""
    for xi, p in zip(x, is_prime):
        if p:
            ax.axvspan(xi - 0.5, xi + 0.5, color='0.92', zorder=0, lw=0)


print('helpers ready')

## 1. Maze graph and its structure
Loaded from the mapping pickle as in the main pipeline. Leaf↔parent links that are virtual (no corridor tile)
are attached from the port layouts and, for other leaves, from the most frequent neighbour in the traversal data.
Roots, corridor and patches are derived from the graph itself (topology-general): the corridor is the edge of
maximal betweenness, its two endpoints are the roots, and each root's other neighbours are patch roots.

In [ ]:
def load_graph(pkl_path):
    """Graph from the mapping pickle, plus each node's position (centre of its tiles) for drawing."""
    import pickle
    with open(pkl_path, 'rb') as f:
        mapping_data = pickle.load(f)
    nodes_dict = mapping_data['mappings']['nodes']
    edges_dict = mapping_data['mappings']['edges']
    G = nx.Graph()
    G.add_nodes_from(nodes_dict.keys())
    for edge_key in edges_dict:
        a, b = edge_key.split('_')
        if a in nodes_dict and b in nodes_dict:
            G.add_edge(a, b)
    pos = {}
    for node, polys in nodes_dict.items():
        pts = [p for poly in (polys or []) for p in poly]
        if pts:
            pos[node] = np.asarray(pts, float).mean(0)
    return G, pos

G_raw, NODE_POS = load_graph(GRAPH_PKL)
print(f'graph from pickle: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges, '
      f'{sum(1 for n in G_raw if G_raw.degree(n) == 0)} isolated nodes')

In [ ]:
def attach_virtual_leaves(G, subjects, traversal_files):
    """Connect isolated nodes: ports from PORT_LAYOUTS, others from traversal neighbours."""
    G = G.copy()
    for subj in subjects:
        for _, _, layout in PORT_LAYOUTS[subj]:
            for leaf, parents in layout.values():
                if leaf in G and parents[0] in G and not G.has_edge(leaf, parents[0]):
                    G.add_edge(leaf, parents[0])
    isolated = {n for n in G if G.degree(n) == 0}
    if isolated:
        counts = {}
        for f in traversal_files:
            t = pd.read_csv(f, usecols=['headstage_graph_node', 'location_type'])
            seq = t.loc[t.location_type == 'node', 'headstage_graph_node'].astype(str).values
            for a, b in zip(seq[:-1], seq[1:]):
                if a in isolated and b not in isolated:
                    counts.setdefault(a, {}).setdefault(b, 0); counts[a][b] += 1
                if b in isolated and a not in isolated:
                    counts.setdefault(b, {}).setdefault(a, 0); counts[b][a] += 1
        for leaf, nb in counts.items():
            G.add_edge(leaf, max(nb, key=nb.get))
    still = [n for n in G if G.degree(n) == 0]
    if still:
        print(f'  {len(still)} nodes never visited and not attached (ignored): {still[:8]}...')
        G.remove_nodes_from(still)
    return G


def maze_structure(G):
    eb = nx.edge_betweenness_centrality(G)
    corridor = max(eb, key=eb.get)
    roots = set(corridor)
    patch_roots = sorted(n for r in roots for n in G.neighbors(r) if n not in roots)
    H = G.copy(); H.remove_nodes_from(roots)
    patch_of = {}
    for k, pr in enumerate(patch_roots):
        for n in nx.node_connected_component(H, pr):
            patch_of[n] = k
    for r in roots:
        patch_of[r] = -1                      # roots / corridor = outside any patch
    return dict(corridor=corridor, roots=roots, patch_roots=patch_roots, patch_of=patch_of,
                dist=dict(nx.all_pairs_shortest_path_length(G)))

_all_traversals = [f for s in SUBJECTS for f in glob.glob(str(PIPELINE_ROOT / s / '*' / FILES['traversal']))]
G = attach_virtual_leaves(G_raw, SUBJECTS, _all_traversals)
MAZE = maze_structure(G)
print(f'graph used: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, tree={nx.is_tree(G)}')
print(f'corridor {MAZE["corridor"]}, patch roots {MAZE["patch_roots"]}')
print('patch sizes:', pd.Series(MAZE['patch_of']).value_counts().sort_index().to_dict())

In [ ]:
def plot_maze(ax, subject):
    """Maze graph at its tile positions; the subject's first-period (maze A) ports coloured by pair."""
    for a, b in G.edges():
        if a in NODE_POS and b in NODE_POS:
            ax.plot(*np.column_stack([NODE_POS[a], NODE_POS[b]]), c='0.75', lw=1.2, zorder=1)
    xy = np.array([NODE_POS[n] for n in G if n in NODE_POS])
    ax.scatter(xy[:, 0], xy[:, 1], s=5, c='0.6', zorder=2)
    for r in MAZE['roots']:
        if r in NODE_POS:
            ax.scatter(*NODE_POS[r], s=60, marker='s', c='k', zorder=3)
    p2pair = pair_of_port(subject)
    for port, (leaf, _) in PORT_LAYOUTS[subject][0][2].items():
        if leaf in NODE_POS:
            ax.scatter(*NODE_POS[leaf], s=110, color=plt.cm.Set1.colors[p2pair[port]], edgecolors='k', zorder=4)
            ax.annotate(str(port), NODE_POS[leaf], ha='center', va='center', fontsize=7, zorder=5)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    ax.set_title(f'{subject}: ports by pair (maze A); squares = roots')

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(5.5 * len(SUBJECTS), 4.5), squeeze=False)
for ax, subj in zip(axes[0], SUBJECTS):
    plot_maze(ax, subj)
fig.legend(handles=[plt.Line2D([], [], marker='o', ls='', color=plt.cm.Set1.colors[k], markeredgecolor='k',
                               label=f'pair {k + 1} (ports {a}, {b})') for k, (a, b) in enumerate(PORT_PAIRS)],
           loc='lower center', ncol=len(PORT_PAIRS))
plt.tight_layout(rect=(0, 0.06, 1, 1))
save_fig(fig, 'fig01_maze_schematic')

## 2. Session discovery
A session is used when the pipeline folder exists and a Bpod `.mat` with a matching trial count is found.

In [ ]:
def find_bpod_mat(subject, ddmmyyyy, n_expected):
    DD, MM, YYYY = ddmmyyyy.split('_')
    hits = sorted((BPOD_DATA_ROOT / subject).glob(f'*/Session Data/{subject}_*_{YYYY}{MM}{DD}_*.mat'))
    if BPOD_PROTOCOL is not None:
        hits = [p for p in hits if p.parent.parent.name == BPOD_PROTOCOL]
    maze = [p for p in hits if 'maze' in p.parent.parent.name.lower()]
    hits = maze or hits
    if not hits:
        return None, None, 'no .mat'
    best, best_diff, note = None, None, ''
    for p in hits:
        S = sio.loadmat(p, squeeze_me=True, struct_as_record=False)['SessionData']
        diff = abs(int(S.nTrials) - n_expected)
        if best_diff is None or diff < best_diff:
            best, best_diff, bestS = p, diff, S
    if best_diff:
        note = f'trial count differs by {best_diff}'
    if len(hits) > 1:
        note += f' ({len(hits)} files; chose {best.name})'
    return best, bestS, note.strip()


SESSIONS = []
for subj in SUBJECTS:
    for d in sorted((PIPELINE_ROOT / subj).glob('*_*_*'), key=lambda p: pd.to_datetime(p.name, format='%d_%m_%Y', errors='coerce')):
        if not re.fullmatch(r'\d{2}_\d{2}_\d{4}', d.name) or not (d / FILES['reward']).exists():
            continue
        rew = pd.read_csv(d / FILES['reward'])
        mat, S, note = find_bpod_mat(subj, d.name, len(rew))
        date = d.name.replace('_', '/')
        SESSIONS.append(dict(subject=subj, folder=d.name, date=date, path=d, mat=mat, S=S, note=note,
                             n_trials=len(rew), maze_prime=is_maze_a_prime(subj, date)))

sess_table = pd.DataFrame([{k: v for k, v in s.items() if k not in ('S', 'path')} for s in SESSIONS])
display(sess_table[['subject', 'date', 'n_trials', 'maze_prime', 'mat', 'note']])
if EXCLUDE_MAZE_A_PRIME:
    SESSIONS = [s for s in SESSIONS if not s['maze_prime']]
SESSIONS = [s for s in SESSIONS if s['S'] is not None]
print(f'{len(SESSIONS)} sessions with Bpod data')

## 3. Trial tables (Bpod + reward_df)
One row per port contact. Active pair per trial comes from `SessionData.Blocks` (raw `.mat`).
Depletion level is reconstructed from consecutive same-port contacts and the session's `DepletionSteps`
(verified exactly against all rewarded NPC1 contacts).

In [ ]:
def build_trials(sess):
    subj, S = sess['subject'], sess['S']
    r = pd.read_csv(sess['path'] / FILES['reward'])
    n = min(len(r), int(S.nTrials))
    if n != len(r) or n != int(S.nTrials):
        print(f"  {subj} {sess['date']}: reward_df {len(r)} vs Bpod {int(S.nTrials)} trials, truncated to {n}")
    r = r.iloc[:n].copy()
    blocks = [tuple(int(p) for p in np.atleast_1d(b)) for b in S.Blocks][:n]
    order = []
    for b in blocks:
        if b not in order:
            order.append(b)
    p2pair = pair_of_port(subj)
    pairs = pairs_for(subj)
    r['port'] = r['reward_port'].astype(int)
    r['pair'] = r['port'].map(p2pair)
    r['active_pair'] = [p2pair[b[0]] for b in blocks]
    r['is_active'] = r['pair'] == r['active_pair']
    bidx = np.r_[0, np.cumsum([blocks[i] != blocks[i - 1] for i in range(1, n)])]
    r['block_index'] = bidx
    r['trial_in_block'] = r.groupby('block_index').cumcount() + 1
    r['block_length'] = r.groupby('block_index')['trial_in_block'].transform('max')
    r['next_pair_in_order'] = [p2pair[order[(order.index(b) + 1) % len(order)][0]] for b in blocks]
    r['rewarded'] = r['reward_size'] > 0
    full = float(np.max(np.atleast_1d(S.TrialRewardSize)))
    n_steps = int(np.max(np.atleast_1d(S.DepletionSteps)))
    k = np.ones(n, int)
    for i in range(1, n):
        k[i] = k[i - 1] + 1 if r['port'].iat[i] == r['port'].iat[i - 1] else 1
    r['contact_k'] = k                                     # k-th consecutive contact at this port
    r['size_now'] = np.clip(full * (1 - (k - 1) / n_steps), 0, None)
    r['size_if_repeat'] = np.clip(full * (1 - k / n_steps), 0, None)
    r['full_size'], r['n_dep_steps'] = full, n_steps
    prev = r['port'].shift()
    r['replenished'] = prev.ne(r['port'])                  # informative contact: port was full
    # absolute Bpod time of each contact (identical to reward_df 'abs_time' where that column exists)
    r['t'] = np.asarray(S.TrialStartTimestamp, float)[:n] + r['reward_time'].astype(float).values
    # frame of each contact: trial start frame (from traversal) + reward_time
    trav = pd.read_csv(sess['path'] / FILES['traversal'])
    tstart = (trav['start_frame_global'] - trav['start_frame_in_trial']).groupby(trav['trial_idx']).min()
    r['frame'] = (r['trial_idx'].map(tstart) + np.round(r['reward_time'] * FPS)).astype('Int64')
    # leaf of each port this session
    layout = resolve_port_layout(subj, sess['date'])
    r['leaf'] = r['port'].map(lambda p: layout[p][0])
    r['subject'], r['date'], r['maze_prime'] = subj, sess['date'], sess['maze_prime']
    r['session'] = f"{subj} {sess['date']}"
    r['session_order'] = SESSIONS.index(sess)
    return r, trav


TRIALS, TRAV = {}, {}
for s in SESSIONS:
    TRIALS[s['date'] + s['subject']], TRAV[s['date'] + s['subject']] = build_trials(s)
trials = pd.concat(TRIALS.values(), ignore_index=True)
print(f'{len(trials)} contacts in {trials.session.nunique()} sessions')
display(trials.groupby('session', sort=False).agg(n=('port', 'size'), correct=('is_active', 'mean'), rewarded=('rewarded', 'mean'),
                                      blocks=('block_index', 'max'), dep_steps=('n_dep_steps', 'first')).round(2))

### 3b. Running reward rate, Bayesian belief and pair values
* `rho_<tau>`: exponentially weighted reward rate (µL-equivalent per second) with time constant tau.
* `belief_<h>`: posterior that the **current** pair is active, from an observer that knows p = 0.9 / 0.1,
  updates only on replenished contacts, and assumes block switches with hazard h per contact to the next pair
  in the Bpod order (`belief_u_<h>` = switch to any other pair uniformly).
* `q_<alpha>_<f>_p<k>`: pair-value Q-learning (cf. Bari et al. 2019; Beron et al. 2022). On a replenished contact at
  pair j, `Q[j] += alpha * (reward - Q[j])`; on every contact the other pairs relax towards 0.5 at rate f (forgetting).
  Unlike the leaky integrator of section 6, values persist across visits, so they can also drive the destination choice.

In [ ]:
def add_running_rate(r, taus=RHO_TAUS_S):
    t, rew = r['t'].values, r['reward_size'].values
    for tau in taus:
        rho = np.zeros(len(r)); acc, t_last = 0.0, t[0]
        for i in range(len(r)):
            acc *= np.exp(-(t[i] - t_last) / tau); t_last = t[i]
            rho[i] = acc / tau                  # rate known BEFORE this contact's outcome
            acc += rew[i]
        r[f'rho_{tau}'] = rho
    return r


def add_belief(r, scales=HAZARD_SCALES):
    npair = int(r['pair'].max()) + 1
    mean_len = r.groupby('block_index')['block_length'].first().mean()
    order_next = {}
    for a, b in zip(r['active_pair'], r['next_pair_in_order']):
        order_next[a] = b
    for sc in scales:
        h = min(sc / mean_len, 0.5)
        for kind in ('o', 'u'):
            T = np.full((npair, npair), 0.0)
            for a in range(npair):
                T[a, a] = 1 - h
                if kind == 'o' and a in order_next:
                    T[a, order_next[a]] += h
                else:
                    for b in range(npair):
                        if b != a:
                            T[a, b] += h / (npair - 1)
            post = np.full(npair, 1.0 / npair)
            bel_after, bel_before = np.zeros(len(r)), np.zeros(len(r))
            for i, row in enumerate(r.itertuples()):
                post = post @ T
                bel_before[i] = post[row.pair]
                if row.replenished:
                    # contact at pair j only tells us about j; other pairs: likelihood equal across hypotheses
                    lik_h = np.array([(P_ACTIVE if row.rewarded else 1 - P_ACTIVE) if a == row.pair
                                      else (P_INACTIVE if row.rewarded else 1 - P_INACTIVE) for a in range(npair)])
                    post = post * lik_h; post /= post.sum()
                bel_after[i] = post[row.pair]
            tag = 'belief' if kind == 'o' else 'belief_u'
            r[f'{tag}_{sc}'] = bel_after
    return r


def add_pair_q(r, alphas=RL_ALPHAS, forgets=RL_FORGET):
    """Pair-value Q-learning with forgetting; values stored after this contact's update, for every pair."""
    pair, rep = r['pair'].values.astype(int), r['replenished'].values
    rew = r['rewarded'].values.astype(float)
    new = {}
    for a in alphas:
        for f in forgets:
            Q, out = np.full(N_PAIRS, 0.5), np.zeros((len(r), N_PAIRS))
            for i in range(len(r)):
                j = pair[i]
                qj = Q[j]
                Q += f * (0.5 - Q)
                Q[j] = qj + a * (rew[i] - qj) if rep[i] else qj
                out[i] = Q
            for k in range(N_PAIRS):
                new[f'q_{a}_{f}_p{k}'] = out[:, k]
    return pd.concat([r, pd.DataFrame(new, index=r.index)], axis=1)


for key in TRIALS:
    TRIALS[key] = add_pair_q(add_belief(add_running_rate(TRIALS[key])))
trials = pd.concat(TRIALS.values(), ignore_index=True)
print('running rate and belief columns:', [c for c in trials.columns if c.startswith(('rho_', 'belief'))])
print(f'pair-value columns: {sum(c.startswith("q_") for c in trials.columns)}')

## 4. Graph-derived variables (NaviGraph)
For each interval between contact i and i+1: node path, path length vs graph shortest path, travel time
(leaving the departure port to arriving at the next leaf), patch-exit onset, aborted exits
(left the patch and came back without another contact), and other patches checked without a contact.

In [ ]:
def node_sequence(trav, maze, max_frames=ZIGZAG_MAX_FRAMES):
    """Node visits (node, start frame, duration) with single-frame x->y->x zigzags collapsed (leaf/parent flicker)."""
    t = trav[trav['location_type'] == 'node']
    seq = [[str(a), int(b), int(c)] for a, b, c in zip(t['headstage_graph_node'], t['start_frame_global'], t['duration'])
           if str(a) in maze['patch_of']]
    changed = True
    while changed:
        changed, out, i = False, [], 0
        while i < len(seq):
            if out and i + 1 < len(seq) and seq[i][2] <= max_frames and out[-1][0] == seq[i + 1][0]:
                out[-1][2] += seq[i][2] + seq[i + 1][2]; i += 2; changed = True
            else:
                if out and out[-1][0] == seq[i][0]:
                    out[-1][2] += seq[i][2]
                else:
                    out.append(list(seq[i]))
                i += 1
        seq = out
    return np.array([s[0] for s in seq]), np.array([s[1] for s in seq])


def interval_features(r, trav, maze):
    nn, nf = node_sequence(trav, maze)
    pof, dist = maze['patch_of'], maze['dist']
    out = []
    for i in range(len(r) - 1):
        f0, f1 = r['frame'].iat[i], r['frame'].iat[i + 1]
        rec = dict(path_len=np.nan, shortest=np.nan, excess=np.nan, travel_s=np.nan, exit_onset_s=np.nan,
                   aborted_exit=False, n_patches_checked=0)
        if pd.isna(f0) or pd.isna(f1) or f1 <= f0:
            out.append(rec); continue
        sel = (nf >= f0) & (nf < f1)
        k0 = np.searchsorted(nf, f0, side='right') - 1          # node occupied at the contact frame
        if k0 >= 0 and not sel[k0]:
            sel[k0] = True
        seq = list(nn[sel]); fr = np.maximum(nf[sel], f0)
        a_leaf, b_leaf = r['leaf'].iat[i], r['leaf'].iat[i + 1]
        if a_leaf in dist and b_leaf in dist[a_leaf]:
            rec['shortest'] = dist[a_leaf][b_leaf]
        if seq:
            compact = [seq[0]] + [s for k, s in enumerate(seq[1:], 1) if s != seq[k - 1]]
            if compact[0] != a_leaf:
                compact = [a_leaf] + compact
            if compact[-1] != b_leaf:
                compact = compact + [b_leaf]
            rec['path_len'] = len(compact) - 1                    # edges traversed, leaf to leaf
            rec['excess'] = rec['path_len'] - rec['shortest'] if a_leaf != b_leaf else np.nan
            home = {a_leaf} | set(G.neighbors(a_leaf)) if a_leaf in G else {a_leaf}
            away = [k for k, s in enumerate(seq) if s not in home]
            if away:
                arr = [k for k, s in enumerate(seq) if s == b_leaf and k >= away[0]]
                f_arr = fr[arr[0]] if arr else f1
                rec['travel_s'] = (f_arr - fr[away[0]]) * FRAME_S
            p_dep = pof.get(a_leaf, -2)
            outside = [k for k, s in enumerate(seq) if pof.get(s, -2) != p_dep]
            if outside:
                rec['exit_onset_s'] = (fr[outside[0]] - f0) * FRAME_S
                if r['pair'].iat[i + 1] == r['pair'].iat[i]:
                    rec['aborted_exit'] = True
            p_arr = pof.get(b_leaf, -2)
            others = {pof[s] for s in seq if pof.get(s, -1) >= 0} - {p_dep, p_arr}
            rec['n_patches_checked'] = len(others)
        out.append(rec)
    out.append(dict(path_len=np.nan, shortest=np.nan, excess=np.nan, travel_s=np.nan, exit_onset_s=np.nan,
                    aborted_exit=False, n_patches_checked=0))
    return pd.DataFrame(out, index=r.index)


for key in TRIALS:
    feats = interval_features(TRIALS[key], TRAV[key], MAZE)
    r = TRIALS[key]
    feats['path_len_notebook'] = feats['path_len']
    if PATH_LENGTH_FROM_PIPELINE and 'reward_port_transition_length_sum' in r:
        moved = r['port'].shift(-1).ne(r['port'])
        feats['path_len'] = r['reward_port_transition_length_sum'].where(moved, np.nan).astype(float)
        feats['excess'] = feats['path_len'] - feats['shortest']
    TRIALS[key] = pd.concat([r.drop(columns=[c for c in feats.columns if c in r]), feats], axis=1)
trials = pd.concat(TRIALS.values(), ignore_index=True)
display(trials.groupby('session', sort=False)[['path_len', 'excess', 'travel_s', 'aborted_exit', 'n_patches_checked']]
        .agg({'path_len': 'median', 'excess': 'median', 'travel_s': 'median', 'aborted_exit': 'mean',
              'n_patches_checked': 'mean'}).round(2))

## 5. Port level: when does the mouse stop repeating? (MVT)
Decision after contact i: **leave** the port (next contact elsewhere) or **repeat**.

MVT (Charnov 1976): leave when the expected gain from repeating falls below what the environment yields on
average over the same time: `E[r_repeat] < rho * t_repeat`. With belief b that the pair is active,
`E[r_repeat] = (b*0.9 + (1-b)*0.1) * size_if_repeat`.

* **MVT (rate threshold)**: DV = rho·t_repeat − E[r_repeat]; t_repeat = session median repeat interval;
  rho = running rate (tau chosen on training sessions among `MVT_ENV_TAUS_S` ≥ 120 s, so that it is an
  environment rate and not a proxy of the local rate).
* **MVT (free weights)**: the two MVT terms enter separately, so the mouse may weigh the opportunity cost and the
  expected gain differently (e.g. subjective travel cost / discounting, Kane et al. 2019).
* **MVT-partner**: compares repeating with moving to the partner port, whose time cost is the
  session median alternation interval (travel + hesitation, from Bpod times).
* Competitors: contact count, **depleted port** (the next repeat yields nothing), win-stay/lose-shift,
  GLM, GLM + depleted.

A session without any repeat (or alternation) has no median interval; it gets the subject's median (this NaN
previously reached every training fold and silently turned all MVT fits into a constant 50 % prediction).

In [ ]:
def port_dataset(trials):
    rows = []
    for s, r in trials.groupby('session', sort=False):
        r = r.reset_index(drop=True)
        nxt = r['port'].shift(-1)
        rep_dt = (r['t'].shift(-1) - r['t'])[nxt == r['port']].median()
        alt_mask = (nxt != r['port']) & (r['pair'].shift(-1) == r['pair'])
        alt_dt = (r['t'].shift(-1) - r['t'])[alt_mask].median()
        d = r.iloc[:-1].copy()
        d['leave_port'] = (nxt.iloc[:-1] != r['port'].iloc[:-1]).astype(int).values
        d['t_repeat'], d['t_alt_bpod'] = rep_dt, alt_dt
        d['dt_since_reward'] = d['t'] - d['t'].where(d['rewarded']).ffill().shift().fillna(d['t'].iloc[0])
        d['depleted'] = (d['size_if_repeat'] <= 0).astype(float)
        rows.append(d)
    out = pd.concat(rows, ignore_index=True)
    for c in ('t_repeat', 't_alt_bpod'):
        missing = out.groupby('session')[c].first().isna()
        if missing.any():
            print(f'  {c}: no such transition in {list(missing[missing].index)}; filled with the subject median')
        out[c] = out[c].fillna(out.groupby('subject')[c].transform('median'))
    return out

port_df = port_dataset(trials)
bel_col = f'belief_{PORT_BELIEF_SCALE}'
port_df['p_rew_if_repeat'] = port_df[bel_col] * P_ACTIVE + (1 - port_df[bel_col]) * P_INACTIVE
port_df['E_repeat'] = port_df['p_rew_if_repeat'] * port_df['size_if_repeat']
port_df['E_partner'] = port_df['p_rew_if_repeat'] * port_df['full_size']


def mvt_X(df):
    return {tau: np.column_stack([df[f'rho_{tau}'] * df['t_repeat'] - df['E_repeat']]) for tau in MVT_ENV_TAUS_S}

def mvt_partner_X(df, travel_col='t_alt_bpod'):
    return np.column_stack([df['E_partner'] / df[travel_col] - df['E_repeat'] / df['t_repeat']])

def glm_port_X(d, extra=()):
    return {tau: np.column_stack([d['contact_k'], d['rewarded'], d['dt_since_reward'], d[f'rho_{tau}']]
                                 + [d[c] for c in extra]) for tau in RHO_TAUS_S}

PORT_MODELS_BPOD = {
    'MVT (rate threshold)': mvt_X,
    'MVT (free weights)': lambda d: {tau: np.column_stack([d[f'rho_{tau}'] * d['t_repeat'], d['E_repeat']])
                                     for tau in MVT_ENV_TAUS_S},
    'MVT partner (Bpod time)': mvt_partner_X,
    'contact count': lambda d: d[['contact_k']].values,
    'depleted port': lambda d: d[['depleted']].values,
    'win-stay/lose-shift': lambda d: d[['rewarded']].values.astype(float),
    'GLM (k, reward, dt, rho)': glm_port_X,
    'GLM + depleted': lambda d: glm_port_X(d, ['depleted']),
}

PORT = {}
for subj in SUBJECTS:
    d = port_df[port_df.subject == subj].reset_index(drop=True)
    print(f'--- {subj}: {len(d)} port decisions in {d.session.nunique()} sessions, P(leave port) = {d.leave_port.mean():.2f}')
    summ, per, pred = loso_compare(d, 'leave_port', PORT_MODELS_BPOD)
    PORT[subj] = dict(df=d, summary=summ, per=per, pred=pred)
    display(summ.round(4))

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 2, figsize=(10, 3.6 * len(SUBJECTS)), squeeze=False)
for row, subj in zip(axes, SUBJECTS):
    R = PORT[subj]; d, pred = R['df'], R['pred']
    best = R['summary']['model'].iat[0]
    n_steps = int(d['n_dep_steps'].mode().iat[0]); kmax = n_steps + 3
    row[0].axvspan(n_steps - 0.5, kmax + 0.5, color='0.93', lw=0, zorder=0)
    for rw, c, lab in [(True, 'tab:green', 'after reward'), (False, 'tab:red', 'after no reward')]:
        m = ((d['rewarded'] == rw) & (d['contact_k'] <= kmax)).values
        plot_binned(row[0], d['contact_k'][m], d['leave_port'][m], d['session'][m],
                    preds={f'{best}, {lab}': (pred[best][m], c)}, color=c, label=f'data, {lab}')
    row[0].set_xlabel('consecutive contact at the port (k)'); row[0].set_ylabel('P(leave port)')
    row[0].set_title(f'{subj}: leaving vs depletion (shaded: next repeat yields nothing)')
    row[0].legend()
    tau = most_common_param(R['per'], 'MVT (rate threshold)')
    dv = d[f'rho_{tau}'] * d['t_repeat'] - d['E_repeat']
    plot_binned(row[1], dv, d['leave_port'], d['session'], bins=10,
                preds={'MVT (rate threshold)': (pred['MVT (rate threshold)'], 'tab:blue'), best: (pred[best], 'tab:orange')})
    row[1].axvline(0, ls='--', c='grey')
    row[1].set_xlabel(f'MVT decision variable  rho·t_repeat − E[r_repeat]  (tau = {tau} s)')
    row[1].set_ylabel('P(leave port)'); row[1].set_title(f'{subj}: MVT decision variable (held-out predictions)')
    row[1].legend()
plt.tight_layout()
save_fig(fig, 'fig02_port_level')

## 6. Patch level: when does the mouse leave the pair? (inference)
Decision after contact i: **leave the patch** (next contact in another pair) or stay (repeat or alternate).
Models:
* **Bayesian observer** (block order known / uniform switches; hazard scale fitted): DV = logit(belief);
  optionally + a bias per pair.
* **Consecutive failures** in the current patch (replenished contacts only), cf. Lottem et al. 2018; Vertechi et al. 2020.
* **Leaky integrator** of outcomes in the current patch, reset on arrival (alpha fitted).
* **Pair-value RL**: Q of the current pair vs mean Q of the other pairs (learning rate and forgetting fitted).
* **MVT patch**: local reward rate in the patch vs running environmental rate, with a fixed (local − rho) or free
  weighting (section 9 reads the implied leaving threshold from the free weights).
* Combinations: leaky + Bayes, leaky + **choice history** (exponential filter of past leave decisions,
  cf. Beron et al. 2022), leaky + **Bpod timing** (giving-up time since the last reward, residence time in the
  patch — both from Bpod timestamps, no tracking needed).

In [ ]:
def patch_dataset(trials, alphas=LEAK_ALPHAS):
    rows = []
    for s, r in trials.groupby('session', sort=False):
        r = r.reset_index(drop=True)
        n = len(r)
        leave = (r['pair'].shift(-1) != r['pair']).astype(int).values
        fails, aborted, local_rate = np.zeros(n), np.zeros(n), np.zeros(n)
        visit_start, last_rew, n_in_visit = np.zeros(n), np.zeros(n), np.zeros(n, int)
        leak = {a: np.zeros(n) for a in alphas}
        hist, h = {a: np.zeros(n) for a in alphas}, {a: 0.0 for a in alphas}
        for i in range(n):
            if i == 0 or r['pair'].iat[i] != r['pair'].iat[i - 1]:      # arrived in a new patch
                c, v, ab, rew_sum, k = 0, {a: 0.5 for a in alphas}, 0, 0.0, 0
                vs = lr = r['t'].iat[i]
            k += 1
            if r['replenished'].iat[i]:
                c = 0 if r['rewarded'].iat[i] else c + 1
                for a in alphas:
                    v[a] = (1 - a) * v[a] + a * float(r['rewarded'].iat[i])
            if r['rewarded'].iat[i]:
                lr = r['t'].iat[i]
            rew_sum += r['reward_size'].iat[i]
            fails[i], aborted[i], visit_start[i], last_rew[i], n_in_visit[i] = c, ab, vs, lr, k
            for a in alphas:
                leak[a][i] = v[a]
                hist[a][i] = h[a]                                       # past leave decisions only
                h[a] = (1 - a) * h[a] + a * leave[i]
            local_rate[i] = rew_sum / (r['t'].iat[i] - vs + HANDLING_S)
            ab += int(bool(r['aborted_exit'].iat[i]))
        d = r.copy()
        d['leave_patch'] = leave
        d['consec_fail'] = fails
        d['contact_in_visit'] = n_in_visit
        for a in alphas:
            d[f'leak_{a}'] = leak[a]
            d[f'leave_hist_{a}'] = hist[a]
        d['residence_s'] = d['t'] - visit_start
        d['giving_up_s'] = d['t'] - last_rew
        d['n_aborted_so_far'] = aborted
        d['local_rate'] = local_rate
        rows.append(d.iloc[:-1])
    return pd.concat(rows, ignore_index=True)

patch_df = patch_dataset(trials)
lg = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))


def bpod_timing(d):
    return np.column_stack([np.log1p(d['giving_up_s'].clip(lower=0)), np.log1p(d['residence_s'].clip(lower=0))])

def pair_dummies(d):
    return np.column_stack([(d['pair'] == k).astype(float) for k in range(1, N_PAIRS)])

def q_features(d, a, f):
    Q = d[[f'q_{a}_{f}_p{k}' for k in range(N_PAIRS)]].values
    cur = Q[np.arange(len(d)), d['pair'].values.astype(int)]
    return np.column_stack([cur, (Q.sum(1) - cur) / (N_PAIRS - 1)])

PATCH_MODELS_BPOD = {
    'Bayes (block order)': lambda d: {sc: lg(d[f'belief_{sc}'].values) for sc in HAZARD_SCALES},
    'Bayes (uniform switch)': lambda d: {sc: lg(d[f'belief_u_{sc}'].values) for sc in HAZARD_SCALES},
    'Bayes (uniform) + pair bias': lambda d: {sc: np.column_stack([lg(d[f'belief_u_{sc}'].values), pair_dummies(d)])
                                              for sc in HAZARD_SCALES},
    'consecutive failures': lambda d: d[['consec_fail']].values,
    'last outcome': lambda d: d[['rewarded']].values.astype(float),
    'leaky integrator': lambda d: {a: d[f'leak_{a}'].values for a in LEAK_ALPHAS},
    'pair-value RL': lambda d: {(a, f): q_features(d, a, f) for a in RL_ALPHAS for f in RL_FORGET},
    'MVT patch (local − rho)': lambda d: {tau: (d['local_rate'] - d[f'rho_{tau}']).values for tau in MVT_ENV_TAUS_S},
    'MVT patch (free weights)': lambda d: {tau: np.column_stack([d['local_rate'], d[f'rho_{tau}']]) for tau in MVT_ENV_TAUS_S},
    'leaky + Bayes (uniform)': lambda d: {(a, sc): np.column_stack([d[f'leak_{a}'], lg(d[f'belief_u_{sc}'].values)])
                                          for a in LEAK_ALPHAS for sc in HAZARD_SCALES},
    'leaky + choice history': lambda d: {(a, b): np.column_stack([d[f'leak_{a}'], d[f'leave_hist_{b}']])
                                         for a in LEAK_ALPHAS for b in LEAK_ALPHAS},
    'leaky + Bpod timing': lambda d: {a: np.column_stack([d[f'leak_{a}'], bpod_timing(d)]) for a in LEAK_ALPHAS},
}

PATCH = {}
for subj in SUBJECTS:
    d = patch_df[patch_df.subject == subj].reset_index(drop=True)
    print(f'--- {subj}: {len(d)} patch decisions, P(leave patch) = {d.leave_patch.mean():.2f}')
    summ, per, pred = loso_compare(d, 'leave_patch', PATCH_MODELS_BPOD)
    PATCH[subj] = dict(df=d, summary=summ, per=per, pred=pred)
    display(summ.round(4))

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(15, 3.8 * len(SUBJECTS)), squeeze=False)
for row, subj in zip(axes, SUBJECTS):
    R = PATCH[subj]; d, pred = R['df'], R['pred']
    best = R['summary']['model'].iat[0]
    a = most_common_param(R['per'], 'leaky integrator')
    sc = most_common_param(R['per'], 'Bayes (uniform switch)')
    m = (d['consec_fail'] <= 6).values
    plot_binned(row[0], d['consec_fail'][m], d['leave_patch'][m], d['session'][m], min_n=15,
                preds={'consecutive failures': (pred['consecutive failures'][m], 'tab:purple'),
                       'leaky integrator': (pred['leaky integrator'][m], 'tab:blue'), best: (pred[best][m], 'tab:orange')})
    row[0].set_xlabel('consecutive failures in the patch'); row[0].set_ylabel('P(leave patch)')
    row[0].set_title(f'{subj}: leaving vs consecutive failures'); row[0].legend()
    plot_binned(row[1], d[f'leak_{a}'], d['leave_patch'], d['session'], bins=10,
                preds={'leaky integrator': (pred['leaky integrator'], 'tab:blue'), best: (pred[best], 'tab:orange')})
    row[1].set_xlabel(f'leaky value of the patch (alpha = {a})'); row[1].set_ylabel('P(leave patch)')
    row[1].set_title(f'{subj}: leaving follows the integrated outcomes'); row[1].legend()
    m = (d['contact_in_visit'] <= 10).values
    plot_binned(row[2], d['contact_in_visit'][m], d['leave_patch'][m], d['session'][m],
                preds={f'Bayes (uniform, h x{sc})': (pred['Bayes (uniform switch)'][m], 'tab:green'),
                       'leaky integrator': (pred['leaky integrator'][m], 'tab:blue'), best: (pred[best][m], 'tab:orange')})
    row[2].set_xlabel('contact number within the visit'); row[2].set_ylabel('P(leave patch)')
    row[2].set_title(f'{subj}: leaving hazard within a visit (posterior check)'); row[2].legend()
plt.tight_layout()
save_fig(fig, 'fig03_patch_level')

### 6b. Model comparison (thesis figure)
Each grey line is one held-out session; the coloured dot is the pooled value of the tables (decision-weighted mean)
with a 95 % bootstrap CI over sessions. Models are in the same order for every subject.

In [ ]:
def shared_order(results):
    """Model order by mean pooled bits across subjects."""
    s = pd.concat([results[subj]['summary'].set_index('model')['bits_per_decision'] for subj in SUBJECTS], axis=1)
    return list(s.mean(1).sort_values(ascending=False).index)

fig, axes = plt.subplots(2, len(SUBJECTS), figsize=(6 * len(SUBJECTS), 10), squeeze=False)
for row, (name, RES) in enumerate([('port level: leave the port?', PORT), ('patch level: leave the pair?', PATCH)]):
    order = shared_order(RES)
    for col, subj in enumerate(SUBJECTS):
        plot_model_comparison(axes[row, col], RES[subj]['per'], order, SUBJECT_COLORS[subj], f'{subj} — {name}')
    lims = [ax.get_ylim() for ax in axes[row]]
    for ax in axes[row]:
        ax.set_ylim(min(l[0] for l in lims), max(l[1] for l in lims))
plt.tight_layout()
save_fig(fig, 'fig04_model_comparison')

overview = pd.concat([RES[subj]['summary'].assign(subject=subj, level=level)
                      for level, RES in [('port', PORT), ('patch', PATCH)] for subj in SUBJECTS])
display(overview.pivot_table(index=['level', 'model'], columns='subject',
                             values=['bits_per_decision', 'pseudo_R2', 'AUC']).round(3))

## 7. Destination after a patch leave
Conditional logit over the other pairs, scored **leave-one-session-out** against a uniform choice (features are
standardised on the training sessions). Candidate features:
* static graph distance between the departure port and the candidate's nearest port (no tracking needed);
* measured travel time between the pairs (session median of direct transits, graph);
* whether the candidate is the next pair in the Bpod block order;
* time since the candidate was last visited (log s), last replenished outcome there (0.5 if never sampled);
* the candidate's pair value Q (pair-value RL, using the subject's most often selected learning rate / forgetting).

In [ ]:
def destination_dataset(trials, q_par):
    rows = []
    for s, r in trials.groupby('session', sort=False):
        r = r.reset_index(drop=True)
        subj = r['subject'].iat[0]; layout = resolve_port_layout(subj, r['date'].iat[0])
        pairs = pairs_for(subj)
        qa, qf = q_par[subj]
        tt = {}                              # measured travel time between pairs (median over direct transitions)
        for i in range(len(r) - 1):
            a, b = r['pair'].iat[i], r['pair'].iat[i + 1]
            if a != b and r['excess'].iat[i] == 0 and not np.isnan(r['travel_s'].iat[i]):
                tt.setdefault((a, b), []).append(r['travel_s'].iat[i])
        tt = {k: np.median(v) for k, v in tt.items()}
        last_t, last_out = {}, {}
        for i in range(len(r) - 1):
            a, b = r['pair'].iat[i], r['pair'].iat[i + 1]
            last_t[a] = r['t'].iat[i]
            if r['replenished'].iat[i]:
                last_out[a] = float(r['rewarded'].iat[i])
            if a == b:
                continue
            for c in range(len(pairs)):
                if c == a:
                    continue
                d_static = min(MAZE['dist'][r['leaf'].iat[i]].get(layout[p][0], np.nan) for p in pairs[c])
                rows.append(dict(session=s, subject=subj, event=f'{s}_{i}', origin=a, cand=c, chosen=int(c == b),
                                 dist=d_static, travel=tt.get((a, c), np.nan),
                                 next_in_order=int(c == r['next_pair_in_order'].iat[i]),
                                 log_time_since_visit=np.log1p(r['t'].iat[i] - last_t[c]) if c in last_t else np.nan,
                                 last_outcome=last_out.get(c, 0.5), q=r[f'q_{qa}_{qf}_p{c}'].iat[i],
                                 was_active=int(c == r['active_pair'].iat[i]), left_active=int(r['is_active'].iat[i])))
    D = pd.DataFrame(rows)
    D['travel'] = D.groupby(['subject', 'cand'])['travel'].transform(lambda x: x.fillna(x.median()))
    D['travel'] = D['travel'].fillna(D['travel'].median())
    never = D['log_time_since_visit'].isna()
    D['log_time_since_visit'] = D.groupby('session')['log_time_since_visit'].transform(lambda x: x.fillna(x.max()))
    D['log_time_since_visit'] = D['log_time_since_visit'].fillna(D['log_time_since_visit'].max())
    print(f'  {never.mean():.1%} of candidates not yet visited in the session (time since visit = session max)')
    return D


def clogit_fit(X, y, idx, ng):
    def nll(b):
        u = X @ b
        lse = np.full(ng, -np.inf); np.logaddexp.at(lse, idx, u)
        return -(u[y == 1].sum() - lse.sum()) + RIDGE * (b ** 2).sum()
    return minimize(nll, np.zeros(X.shape[1]), method='L-BFGS-B').x


def clogit_event_ll(b, X, y, idx, ng):
    """Log-probability of the chosen candidate per event (rows of an event are contiguous, one chosen)."""
    u = X @ b
    lse = np.full(ng, -np.inf); np.logaddexp.at(lse, idx, u)
    return u[y == 1] - lse


def standardise(X, mu=None, sd=None):
    if mu is None:
        mu, sd = X.mean(0), X.std(0) + 1e-9
    return (X - mu) / sd, mu, sd


def loso_clogit(D, models):
    rows = []
    for s in D['session'].unique():
        tr, te = D[D.session != s], D[D.session == s]
        rec = {'session': s, 'n': te['event'].nunique(), 'll_uniform': -np.log(te.groupby('event').size()).sum()}
        for m, cols in models.items():
            Xtr, mu, sd = standardise(tr[cols].values.astype(float))
            Xte = standardise(te[cols].values.astype(float), mu, sd)[0]
            b = clogit_fit(Xtr, tr['chosen'].values, pd.factorize(tr['event'])[0], tr['event'].nunique())
            rec[f'll_{m}'] = clogit_event_ll(b, Xte, te['chosen'].values, pd.factorize(te['event'])[0],
                                             te['event'].nunique()).sum()
        rows.append(rec)
    per = pd.DataFrame(rows)
    n = per['n'].sum()
    summary = pd.DataFrame({
        'model': list(models),
        'bits_vs_uniform': [(per[f'll_{m}'] - per['ll_uniform']).sum() / n / np.log(2) for m in models],
        'sessions_better_than_uniform': [int((per[f'll_{m}'] > per['ll_uniform']).sum()) for m in models],
        'n_sessions': len(per), 'n_events': n,
    }).sort_values('bits_vs_uniform', ascending=False).reset_index(drop=True)
    return summary, per


DEST_MODELS = {
    'graph distance': ['dist'],
    'measured travel (graph)': ['travel'],
    'next in block order': ['next_in_order'],
    'time since last visit': ['log_time_since_visit'],
    'last outcome at pair': ['last_outcome'],
    'pair value Q (RL)': ['q'],
    'distance + recency + last outcome': ['dist', 'log_time_since_visit', 'last_outcome'],
    'full (distance, recency, Q, order)': ['dist', 'log_time_since_visit', 'q', 'next_in_order'],
}
FULL_DEST = 'full (distance, recency, Q, order)'

q_par = {subj: most_common_param(PATCH[subj]['per'], 'pair-value RL') for subj in SUBJECTS}
dest_df = destination_dataset(trials, q_par)
DEST = {}
for subj in SUBJECTS:
    D = dest_df[dest_df.subject == subj].reset_index(drop=True)
    summ, per = loso_clogit(D, DEST_MODELS)
    cols = DEST_MODELS[FULL_DEST]
    b = clogit_fit(standardise(D[cols].values.astype(float))[0], D['chosen'].values,
                   pd.factorize(D['event'])[0], D['event'].nunique())
    ch = D[D.chosen == 1]
    DEST[subj] = dict(df=D, summary=summ, per=per, coef=pd.DataFrame({'feature': cols, 'coef_standardised': b}))
    print(f'--- {subj}: {D.event.nunique()} patch leaves; Q params (alpha, forget) = {q_par[subj]}')
    print(f'  P(destination = next pair in block order) = {ch.next_in_order.mean():.3f} (chance {1 / (N_PAIRS - 1):.2f}); '
          f'P(destination = active pair | left an inactive pair) = {ch[ch.left_active == 0].was_active.mean():.3f}')
    display(summ.round(4)); display(DEST[subj]['coef'].round(3))

In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(15, 4 * len(SUBJECTS)), squeeze=False,
                         gridspec_kw=dict(width_ratios=[1, 1, 1.3]))
for row, subj in zip(axes, SUBJECTS):
    D = DEST[subj]['df']; ch = D[D.chosen == 1]
    M = pd.crosstab(ch['origin'], ch['cand'], normalize='index').reindex(index=range(N_PAIRS), columns=range(N_PAIRS))
    im = row[0].imshow(M.values, cmap='viridis', vmin=0, vmax=1)
    for i in range(N_PAIRS):
        for j in range(N_PAIRS):
            if i != j and np.isfinite(M.values[i, j]):
                row[0].text(j, i, f'{M.values[i, j]:.2f}', ha='center', va='center', fontsize=8,
                            color='w' if M.values[i, j] < 0.6 else 'k')
    row[0].set_xticks(range(N_PAIRS), [f'pair {k + 1}' for k in range(N_PAIRS)])
    row[0].set_yticks(range(N_PAIRS), [f'pair {k + 1}' for k in range(N_PAIRS)])
    row[0].set_xlabel('destination'); row[0].set_ylabel('origin')
    row[0].set_title(f'{subj}: P(destination | origin), chance {1 / (N_PAIRS - 1):.2f}')
    plt.colorbar(im, ax=row[0], fraction=0.046)
    C = DEST[subj]['coef']
    row[1].barh(C['feature'], C['coef_standardised'], color=SUBJECT_COLORS[subj])
    row[1].axvline(0, c='k', lw=0.6); row[1].invert_yaxis()
    row[1].set_xlabel('standardised weight (all sessions)'); row[1].set_title(f'{subj}: what drives the destination')
    plot_model_comparison(row[2], DEST[subj]['per'], list(DEST[subj]['summary']['model']), SUBJECT_COLORS[subj],
                          f'{subj}: held-out destination models', base='ll_uniform')
    row[2].set_ylabel('held-out bits / leave\nvs uniform')
plt.tight_layout()
save_fig(fig, 'fig05_destination')

## 8. Graph-upgraded models
What the NaviGraph tracking adds on top of Bpod:
* Port level: MVT-partner with the **measured** travel time to the partner port (median direct transit, graph)
  plus the consumption period, instead of the Bpod inter-contact interval.
* Patch level: **aborted exits** so far in the visit (left the patch and came back without another contact), and the
  **expected travel time** of leaving from the current pair (session median of the measured direct-path travel
  times out of this pair; prospective, unlike the travel of the leave actually taken).

Giving-up and residence times are Bpod-derived and are part of the Bpod baseline here.

In [ ]:
def add_graph_travel(port_df, trials):
    out = []
    for s, d in port_df.groupby('session', sort=False):
        r = trials[trials.session == s].reset_index(drop=True)
        nxt_same_pair = (r['pair'].shift(-1) == r['pair']) & (r['port'].shift(-1) != r['port'])
        direct = nxt_same_pair & (r['excess'] == 0)
        d = d.copy(); d['t_alt_graph'] = r.loc[direct, 'travel_s'].median() + CONSUMPTION_S
        out.append(d)
    out = pd.concat(out)
    out['t_alt_graph'] = out['t_alt_graph'].fillna(out.groupby('subject')['t_alt_graph'].transform('median'))
    return out


def add_expected_travel(patch_df, trials):
    out = []
    for s, d in patch_df.groupby('session', sort=False):
        r = trials[trials.session == s].reset_index(drop=True)
        direct_leave = (r['pair'].shift(-1) != r['pair']) & (r['excess'] == 0)
        d = d.copy(); d['expected_travel_s'] = d['pair'].map(r.loc[direct_leave].groupby('pair')['travel_s'].median())
        out.append(d)
    out = pd.concat(out)
    out['expected_travel_s'] = out['expected_travel_s'].fillna(out.groupby('subject')['expected_travel_s'].transform('median'))
    return out


port_df = add_graph_travel(port_df, trials)
patch_df = add_expected_travel(patch_df, trials)
log_travel = lambda d: np.log(d['expected_travel_s'].clip(lower=0.1))

PORT_MODELS_GRAPH = {
    'MVT partner (Bpod time)': mvt_partner_X,
    'MVT partner (graph travel)': lambda d: mvt_partner_X(d, 't_alt_graph'),
    'GLM + depleted': PORT_MODELS_BPOD['GLM + depleted'],
    'GLM + depleted + graph travel': lambda d: glm_port_X(d, ['depleted', 't_alt_graph']),
}
PATCH_MODELS_GRAPH = {
    'leaky + Bpod timing': PATCH_MODELS_BPOD['leaky + Bpod timing'],
    'leaky + Bpod timing + aborted exits': lambda d: {a: np.column_stack([d[f'leak_{a}'], bpod_timing(d), d['n_aborted_so_far']])
                                                      for a in LEAK_ALPHAS},
    'leaky + Bpod timing + expected travel': lambda d: {a: np.column_stack([d[f'leak_{a}'], bpod_timing(d), log_travel(d)])
                                                        for a in LEAK_ALPHAS},
    'MVT patch (free weights)': PATCH_MODELS_BPOD['MVT patch (free weights)'],
    'MVT patch + expected travel': lambda d: {tau: np.column_stack([d['local_rate'], d[f'rho_{tau}'], log_travel(d)])
                                              for tau in MVT_ENV_TAUS_S},
}
GRAPH_COMPARISONS = [   # (label, level, graph model, its Bpod counterpart)
    ('port: MVT partner,\ngraph vs Bpod time', 'port', 'MVT partner (graph travel)', 'MVT partner (Bpod time)'),
    ('port: GLM\n+ graph travel', 'port', 'GLM + depleted + graph travel', 'GLM + depleted'),
    ('patch: leaky + timing\n+ aborted exits', 'patch', 'leaky + Bpod timing + aborted exits', 'leaky + Bpod timing'),
    ('patch: leaky + timing\n+ expected travel', 'patch', 'leaky + Bpod timing + expected travel', 'leaky + Bpod timing'),
    ('patch: MVT\n+ expected travel', 'patch', 'MVT patch + expected travel', 'MVT patch (free weights)'),
]

GRAPH_PORT, GRAPH_PATCH = {}, {}
for subj in SUBJECTS:
    print(f'--- {subj}')
    for store, df_all, y, models in [(GRAPH_PORT, port_df, 'leave_port', PORT_MODELS_GRAPH),
                                     (GRAPH_PATCH, patch_df, 'leave_patch', PATCH_MODELS_GRAPH)]:
        d = df_all[df_all.subject == subj].reset_index(drop=True)
        summ, per, pred = loso_compare(d, y, models)
        store[subj] = dict(df=d, summary=summ, per=per, pred=pred)
        display(summ.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(5.5 * len(SUBJECTS), 4.2), squeeze=False, sharey=True)
for ax, subj in zip(axes[0], SUBJECTS):
    for x, (_, level, gm, bm) in enumerate(GRAPH_COMPARISONS):
        per = (GRAPH_PORT if level == 'port' else GRAPH_PATCH)[subj]['per']
        delta = (per[f'll_{gm}'] - per[f'll_{bm}']) / per['n'] / np.log(2)
        ax.scatter(x + rng.uniform(-0.15, 0.15, len(delta)), delta, s=8, c='0.6', zorder=2)
        pooled = (per[f'll_{gm}'] - per[f'll_{bm}']).sum() / per['n'].sum() / np.log(2)
        ax.plot([x - 0.3, x + 0.3], [pooled] * 2, c=SUBJECT_COLORS[subj], lw=3, zorder=3)
        ax.text(x + 0.32, pooled, f'{pooled:+.4f}', fontsize=7, va='center')
    ax.axhline(0, c='k', lw=0.6, ls='--')
    ax.set_xticks(range(len(GRAPH_COMPARISONS)), [c[0] for c in GRAPH_COMPARISONS], rotation=40, ha='right', fontsize=7)
    ax.set_ylabel('held-out Δ bits / decision\n(graph model − Bpod counterpart)')
    ax.set_title(f'{subj}: what the tracking adds (dots = sessions, bar = pooled)')
plt.tight_layout()
save_fig(fig, 'fig06_graph_contribution')

## 9. MVT threshold test at patch leaves
MVT predicts that the forager leaves when the local intake rate falls to the environmental rate, and that a longer
travel time lowers the leaving threshold (stay longer).
* **Implied threshold slope**: from the free-weight MVT patch model, `−β_rho / β_local` is how much the local rate at
  which leaving becomes likely rises per unit of environmental rate; MVT predicts 1 (in these units, up to the
  handling-time convention in the local rate).
* **Ratio at leaves**: local rate at the moment of leaving / environmental rate, for leaves after ≥ 1 reward in the
  visit. > 1 = leaving earlier than MVT (under-harvesting); < 1 = staying longer (over-harvesting, the usual finding).
* **Travel**: local rate at leaving vs the **expected** (prospective) travel time out of the current pair.

In [ ]:
MVT_TEST = {}
fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(14, 3.8 * len(SUBJECTS)), squeeze=False)
for row, subj in zip(axes, SUBJECTS):
    d = GRAPH_PATCH[subj]['df'].copy()
    tau = most_common_param(PATCH[subj]['per'], 'MVT patch (free weights)')
    b = fit_logit(np.column_stack([d['local_rate'], d[f'rho_{tau}']]), d['leave_patch'].values.astype(float))
    slope = -b[2] / b[1]
    visit = ((d['pair'] != d['pair'].shift()) | (d['session'] != d['session'].shift())).cumsum()
    d['visit_rewards'] = d.groupby(visit)['rewarded'].cumsum()
    lv = d[(d.leave_patch == 1) & (d.visit_rewards >= 1)]
    ratio = (lv['local_rate'] / lv[f'rho_{tau}']).replace([np.inf, -np.inf], np.nan).dropna()
    ok = lv[['expected_travel_s', 'local_rate']].dropna()
    rs = spearmanr(ok['expected_travel_s'], ok['local_rate'])
    MVT_TEST[subj] = dict(tau_s=tau, beta_local=b[1], beta_rho=b[2], threshold_slope=slope, n_leaves=len(lv),
                          median_ratio=ratio.median(), ratio_q25=ratio.quantile(0.25), ratio_q75=ratio.quantile(0.75),
                          frac_ratio_above_1=(ratio > 1).mean(),
                          spearman_travel_vs_threshold=rs.statistic, spearman_p=rs.pvalue)
    print(f'{subj}: tau = {tau} s, implied threshold slope = {slope:.2f} (MVT: 1); {len(lv)} leaves after >= 1 reward, '
          f'median local/environment = {ratio.median():.2f}; Spearman(expected travel, local rate at leave) = '
          f'{rs.statistic:.2f} (p = {rs.pvalue:.2g})')
    row[0].hist(np.log2(ratio.clip(lower=1 / 64, upper=64)), bins=40, color=SUBJECT_COLORS[subj], alpha=0.8)
    row[0].axvline(0, c='k', ls='--', lw=1); row[0].axvline(np.log2(ratio.median()), c='k', lw=1.5)
    row[0].set_xlabel('log2(local rate at leave / environment rate)'); row[0].set_ylabel('patch leaves')
    row[0].set_title(f'{subj}: leave threshold vs MVT (dashed = MVT)')
    plot_binned(row[1], ok['expected_travel_s'], ok['local_rate'], lv.loc[ok.index, 'session'], bins=6,
                color=SUBJECT_COLORS[subj])
    row[1].set_xlabel('expected travel time out of the pair (s, graph)'); row[1].set_ylabel('local rate at leave')
    row[1].set_title(f'{subj}: Spearman {rs.statistic:.2f} (p = {rs.pvalue:.2g}); MVT predicts < 0')
    pred = PATCH[subj]['pred']
    plot_binned(row[2], d['local_rate'] - d[f'rho_{tau}'], d['leave_patch'], d['session'], bins=10,
                preds={'MVT patch (local − rho)': (pred['MVT patch (local − rho)'], 'tab:blue'),
                       'MVT patch (free weights)': (pred['MVT patch (free weights)'], 'tab:orange')})
    row[2].axvline(0, c='grey', ls='--')
    row[2].set_xlabel(f'local rate − environment rate (tau = {tau} s)'); row[2].set_ylabel('P(leave patch)')
    row[2].set_title(f'{subj}: implied threshold slope {slope:.2f}'); row[2].legend()
plt.tight_layout()
save_fig(fig, 'fig07_mvt_threshold')
display(pd.DataFrame(MVT_TEST).T.round(3))

## 10. Across sessions: model fit and fitted parameters
Top: held-out bits per decision for each session (from the leave-one-session-out runs above).
Bottom: parameters fitted **within each session** (grid choice by in-session likelihood): leaky-integrator learning
rate and Bayesian hazard scale (middle), and the MVT implied threshold slope (bottom). Change-of-maze (A′) sessions are shaded.

In [ ]:
def fit_in_session(d, make, y_col):
    """Best grid parameter and coefficients of one model within one session (in-sample)."""
    y = d[y_col].values.astype(float)
    best = None
    for par, X in as_grid(make, d).items():
        if not np.isfinite(X).all():
            continue
        b = fit_logit(X, y); ll = logit_ll(b, X, y).sum()
        if best is None or ll > best[0]:
            best = (ll, par, b)
    return best[1], best[2]

maze_prime_of = trials.groupby('session')['maze_prime'].first()
PARAMS = {}
for subj in SUBJECTS:
    d_all = PATCH[subj]['df']
    tau = MVT_TEST[subj]['tau_s']
    rows = []
    for s, d in d_all.groupby('session', sort=False):
        if d['leave_patch'].nunique() < 2 or len(d) < 30:
            continue
        a, _ = fit_in_session(d, PATCH_MODELS_BPOD['leaky integrator'], 'leave_patch')
        sc, _ = fit_in_session(d, PATCH_MODELS_BPOD['Bayes (uniform switch)'], 'leave_patch')
        _, b = fit_in_session(d, lambda x: np.column_stack([x['local_rate'], x[f'rho_{tau}']]), 'leave_patch')
        rows.append(dict(session=s, date=session_dates([s])[0], maze_prime=maze_prime_of[s], n=len(d),
                         leaky_alpha=a, bayes_hazard_scale=sc, mvt_threshold_slope=-b[2] / b[1]))
    PARAMS[subj] = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

fig, axes = plt.subplots(3, len(SUBJECTS), figsize=(6.5 * len(SUBJECTS), 10.5), squeeze=False, sharex='col')
for col, subj in enumerate(SUBJECTS):
    ax = axes[0, col]
    lines = [('port', PORT, PORT[subj]['summary']['model'].iat[0]), ('patch', PATCH, 'leaky integrator')]
    if PATCH[subj]['summary']['model'].iat[0] != 'leaky integrator':
        lines.append(('patch', PATCH, PATCH[subj]['summary']['model'].iat[0]))
    sess = PATCH[subj]['per']['session'].values
    sess = sess[np.argsort(session_dates(sess))]
    x = np.arange(len(sess))
    shade_maze_prime(ax, x, maze_prime_of[sess].values)
    for (level, RES, m), c in zip(lines, ['tab:blue', 'tab:green', 'tab:orange']):
        per = RES[subj]['per'].set_index('session').reindex(sess).reset_index()
        ax.plot(x, session_bits(per, m), 'o-', ms=3, c=c, label=f'{level}: {m}')
    ax.axhline(0, c='k', lw=0.6, ls='--')
    ax.set_ylabel('held-out bits / decision')
    ax.set_title(f'{subj}: model fit across sessions (shaded = maze A′)'); ax.legend()
    ax = axes[1, col]; P = PARAMS[subj]; x = np.arange(len(P))
    shade_maze_prime(ax, x, P['maze_prime'].values)
    ax.plot(x, P['leaky_alpha'], 'o-', ms=3, c='tab:blue', label='leaky alpha')
    ax.plot(x, P['bayes_hazard_scale'], 's-', ms=3, c='tab:green', label='Bayes hazard scale')
    for v in (min(LEAK_ALPHAS), max(LEAK_ALPHAS), min(HAZARD_SCALES), max(HAZARD_SCALES)):
        ax.axhline(v, c='0.8', lw=0.5, ls=':')
    ax.set_yscale('log'); ax.set_ylabel('fitted parameter (log;\ndotted = grid limits)')
    ax.set_title(f'{subj}: within-session parameters'); ax.legend()
    ax = axes[2, col]
    shade_maze_prime(ax, x, P['maze_prime'].values)
    ax.plot(x, P['mvt_threshold_slope'].clip(-5, 10), '^-', ms=3, c='tab:red')
    ax.axhline(1, c='k', ls='--', lw=1, label='MVT prediction'); ax.axhline(0, c='0.6', lw=0.6)
    ax.set_xlabel('session (chronological)'); ax.set_ylabel('implied threshold slope\n(clipped to [-5, 10])')
    ax.set_title(f'{subj}: MVT threshold slope per session (tau = {tau} s)'); ax.legend()
plt.tight_layout()
save_fig(fig, 'fig08_across_sessions')
for subj in SUBJECTS:
    sb = session_bits(PATCH[subj]['per'], 'leaky integrator')
    rs = spearmanr(session_dates(PATCH[subj]['per']['session']).astype(float), sb)
    print(f'{subj}: trend of patch-level leaky fit over sessions, Spearman {rs.statistic:.2f} (p = {rs.pvalue:.2g})')

## 11. Engagement states: GLM-HMM (Ashwood et al. 2022)
The patch-leave decision is modelled as a logistic regression whose weights switch between K hidden states that
follow a Markov chain across decisions within a session (features: leaky value, log giving-up time, log residence
time, standardised per subject). K = 1 is plain logistic regression. Fitted by EM; held-out score = one-step-ahead
predictive likelihood of held-out sessions (forward filter), with `GLMHMM_FOLDS` session-wise folds
(interleaved in time). Single initialisation (seeded); states are ordered by their baseline leave tendency.

In [ ]:
def hmm_forward_backward(B, A, pi):
    """Scaled forward-backward. B: (T, K) observation likelihoods. Returns gamma, summed xi, log-likelihood."""
    T, K = B.shape
    al, c = np.empty((T, K)), np.empty(T)
    a = pi * B[0]; c[0] = a.sum(); al[0] = a / c[0]
    for t in range(1, T):
        a = (al[t - 1] @ A) * B[t]; c[t] = a.sum(); al[t] = a / c[t]
    be = np.ones((T, K))
    for t in range(T - 2, -1, -1):
        be[t] = A @ (B[t + 1] * be[t + 1]) / c[t + 1]
    xi = (al[:-1, :, None] * A[None] * (B[1:] * be[1:])[:, None, :] / c[1:, None, None]).sum(0)
    return al * be, xi, np.log(c).sum()


def glmhmm_lik(model, X, y):
    return np.exp(np.column_stack([logit_ll(b, X, y) for b in model['betas']]))


def glmhmm_fit(Xs, ys, K, n_iter=GLMHMM_MAX_ITER, tol=1e-5, seed=RANDOM_SEED):
    X, y = np.vstack(Xs), np.concatenate(ys)
    b0 = fit_logit(X, y)
    g = np.random.default_rng(seed)
    model = dict(betas=[b0 + (g.normal(0, 0.5, b0.shape) if K > 1 else 0) for _ in range(K)],
                 A=np.full((K, K), 0.05 / max(K - 1, 1)) + np.eye(K) * (0.95 - 0.05 / max(K - 1, 1)) if K > 1 else np.ones((1, 1)),
                 pi=np.full(K, 1 / K))
    prev = -np.inf
    for _ in range(n_iter):
        G, XI, P0, LL = [], np.zeros((K, K)), np.zeros(K), 0.0
        for Xi, yi in zip(Xs, ys):
            gm, xi, ll = hmm_forward_backward(glmhmm_lik(model, Xi, yi), model['A'], model['pi'])
            G.append(gm); XI += xi; P0 += gm[0]; LL += ll
        G = np.vstack(G)
        model['betas'] = [fit_logit(X, y, w=G[:, k]) for k in range(K)]
        model['A'] = (XI + 1) / (XI + 1).sum(1, keepdims=True)     # Dirichlet(2) prior keeps rows proper
        model['pi'] = (P0 + 1) / (P0 + 1).sum()
        if LL - prev < tol * abs(LL):
            break
        prev = LL
    order = np.argsort([b[0] for b in model['betas']])             # state 0 = lowest baseline leave tendency
    model['betas'] = [model['betas'][k] for k in order]
    model['A'] = model['A'][np.ix_(order, order)]; model['pi'] = model['pi'][order]
    model['ll'] = LL
    return model


def glmhmm_predictive_ll(model, X, y):
    """log p(y_1..T) = sum_t log p(y_t | y_<t): forward filter only, no look-ahead."""
    B = glmhmm_lik(model, X, y)
    a, total = model['pi'] * B[0], 0.0
    total += np.log(a.sum()); a /= a.sum()
    for t in range(1, len(y)):
        a = (a @ model['A']) * B[t]; s = a.sum(); total += np.log(s); a /= s
    return total


GLMHMM = {}
for subj in SUBJECTS:
    d = PATCH[subj]['df']
    a = most_common_param(PATCH[subj]['per'], 'leaky integrator')
    F = np.column_stack([d[f'leak_{a}'], bpod_timing(d)])
    F = (F - F.mean(0)) / F.std(0)
    y, sess = d['leave_patch'].values.astype(float), d['session'].values
    order = list(pd.Series(pd.unique(sess)).iloc[np.argsort(session_dates(pd.unique(sess)))])
    Xs, ys = {s: F[sess == s] for s in order}, {s: y[sess == s] for s in order}
    fold = {s: i % GLMHMM_FOLDS for i, s in enumerate(order)}
    rows = {s: {'session': s, 'n': len(ys[s])} for s in order}
    for f in range(GLMHMM_FOLDS):
        tr = [s for s in order if fold[s] != f]
        te = [s for s in order if fold[s] == f]
        p0 = np.clip(np.concatenate([ys[s] for s in tr]).mean(), 1e-6, 1 - 1e-6)
        for K in GLMHMM_STATES:
            mdl = glmhmm_fit([Xs[s] for s in tr], [ys[s] for s in tr], K)
            for s in te:
                rows[s][f'll_K={K}'] = glmhmm_predictive_ll(mdl, Xs[s], ys[s])
        for s in te:
            rows[s]['ll_const'] = (ys[s] * np.log(p0) + (1 - ys[s]) * np.log(1 - p0)).sum()
    per = pd.DataFrame(rows.values())
    n = per['n'].sum()
    summary = pd.DataFrame({'model': [f'K={K}' for K in GLMHMM_STATES],
                            'bits_per_decision': [session_bits(per, f'K={K}').mul(per['n']).sum() / n for K in GLMHMM_STATES],
                            'sessions_better_than_K1': [int((per[f'll_K={K}'] > per['ll_K=1']).sum()) for K in GLMHMM_STATES],
                            'n_sessions': len(per)})
    full = {K: glmhmm_fit(list(Xs.values()), list(ys.values()), K) for K in GLMHMM_STATES}
    GLMHMM[subj] = dict(per=per, summary=summary, full=full, Xs=Xs, ys=ys, order=order,
                        features=['intercept', f'leaky value (a={a})', 'log giving-up time', 'log residence time'])
    print(f'--- {subj}'); display(summary.round(4))
    for K in GLMHMM_STATES[1:]:
        print(f'  K={K} transition matrix:\n', np.round(full[K]['A'], 3))

In [ ]:
K_SHOW = 2 if 2 in GLMHMM_STATES else max(GLMHMM_STATES)
state_colors = plt.cm.Dark2.colors
fig, axes = plt.subplots(len(SUBJECTS), 4, figsize=(18, 3.8 * len(SUBJECTS)), squeeze=False,
                         gridspec_kw=dict(width_ratios=[0.8, 1, 1.6, 1.1]))
for row, subj in zip(axes, SUBJECTS):
    H = GLMHMM[subj]
    plot_model_comparison(row[0], H['per'], [f'K={K}' for K in GLMHMM_STATES], SUBJECT_COLORS[subj], f'{subj}: held-out fit')
    mdl = H['full'][K_SHOW]
    W = np.array(mdl['betas']); x = np.arange(W.shape[1]); w = 0.8 / K_SHOW
    for k in range(K_SHOW):
        row[1].bar(x + (k - (K_SHOW - 1) / 2) * w, W[k], w, color=state_colors[k], label=f'state {k + 1}')
    row[1].axhline(0, c='k', lw=0.6)
    row[1].set_xticks(x, H['features'], rotation=30, ha='right', fontsize=7)
    row[1].set_ylabel('weight (standardised)'); row[1].set_title(f'{subj}: GLM weights per state (K={K_SHOW})'); row[1].legend()
    occ = []
    for s in H['order']:
        gm, _, _ = hmm_forward_backward(glmhmm_lik(mdl, H['Xs'][s], H['ys'][s]), mdl['A'], mdl['pi'])
        occ.append(gm.mean(0))
    ex = max(H['order'], key=lambda s: len(H['ys'][s]))
    gm, _, _ = hmm_forward_backward(glmhmm_lik(mdl, H['Xs'][ex], H['ys'][ex]), mdl['A'], mdl['pi'])
    for k in range(K_SHOW):
        row[2].plot(gm[:, k], c=state_colors[k], lw=1, label=f'P(state {k + 1})')
    lv = np.where(H['ys'][ex] == 1)[0]
    row[2].plot(lv, np.full(len(lv), 1.05), '|', c='k', ms=5, label='patch leave')
    row[2].set_xlabel('decision in session'); row[2].set_ylabel('posterior state probability')
    row[2].set_title(f'{subj}: example session {ex.split()[1]}'); row[2].legend(loc='lower right')
    occ = np.array(occ); xs = np.arange(len(occ))
    shade_maze_prime(row[3], xs, maze_prime_of[H['order']].values)
    row[3].stackplot(xs, occ.T, colors=state_colors[:K_SHOW], alpha=0.85)
    row[3].set_xlabel('session (chronological)'); row[3].set_ylabel('fraction of decisions')
    row[3].set_title(f'{subj}: state occupancy across sessions')
plt.tight_layout()
save_fig(fig, 'fig09_glmhmm_states')

## 12. Save tables
Per subject in `Graphs/<subject>/foraging_models/`; the cross-subject overview in `Graphs/all_subjects/foraging_models/`.
Figures were saved as they were made (section headers above).

In [ ]:
for subj in SUBJECTS:
    out = OUTPUT_ROOT / subj / 'foraging_models'
    out.mkdir(parents=True, exist_ok=True)
    trials[trials.subject == subj].to_csv(out / 'trials_with_model_variables.csv', index=False)
    tables = {
        'port_models_bpod': PORT[subj]['summary'], 'port_per_session': PORT[subj]['per'],
        'patch_models_bpod': PATCH[subj]['summary'], 'patch_per_session': PATCH[subj]['per'],
        'port_models_graph': GRAPH_PORT[subj]['summary'], 'port_graph_per_session': GRAPH_PORT[subj]['per'],
        'patch_models_graph': GRAPH_PATCH[subj]['summary'], 'patch_graph_per_session': GRAPH_PATCH[subj]['per'],
        'destination_models': DEST[subj]['summary'], 'destination_per_session': DEST[subj]['per'],
        'destination_coefficients': DEST[subj]['coef'],
        'mvt_threshold': pd.DataFrame([MVT_TEST[subj]]),
        'params_per_session': PARAMS[subj],
        'glmhmm_models': GLMHMM[subj]['summary'], 'glmhmm_per_session': GLMHMM[subj]['per'],
    }
    for name, tab in tables.items():
        tab.to_csv(out / f'{name}.csv', index=False)
    print(f'saved {len(tables) + 1} tables to {out.resolve()}')

out = OUTPUT_ROOT / 'all_subjects' / 'foraging_models'
out.mkdir(parents=True, exist_ok=True)
pd.concat([R[subj]['summary'].assign(subject=subj, level=level)
           for level, R in [('port', PORT), ('patch', PATCH), ('port_graph', GRAPH_PORT), ('patch_graph', GRAPH_PATCH)]
           for subj in SUBJECTS]).to_csv(out / 'model_overview.csv', index=False)
print(f'overview and figures in {out.resolve()}')